# A5：混合模型路由 — Semantic Router + Shadow Evaluation

> **對應 Blog：** FDE 面試準備指南（二十四）混合模型路由與語意路由器設計  
> **核心目標：** 用 Embedding 實作真實的 Semantic Router，看到路由決策邏輯，並建立 Shadow Eval 流程驗證路由品質。

## 面試情境

> 「客戶希望設計一個 Hybrid Model Routing 系統：70% 的簡單日常問候和格式轉換自動路由到小模型；複雜推理和多步驟 Tool-calling 才路由到 Gemini 1.5 Pro。你如何設計這個路由器？如何用統計評估確保路由器不會因為誤判讓整體品質下滑？」

## 學習路徑

| Part | 內容 | 關鍵觀察 |
|------|------|----------|
| 1 | 問題與成本計算 | 量化節省 |
| 2 | Layer 1：Rule-based 路由 | < 1ms 快速過濾 |
| 3 | Layer 2：Semantic Router | Embedding 相似度路由 |
| 4 | Shadow Evaluation Pipeline | 驗證路由品質 |
| 5 | 動態閾值調整 | 自動修正誤判 |
| 6 | 面試白板語言 | |

In [ ]:
import os
import json
import asyncio
import time
import numpy as np
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

from openai import AsyncOpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

client = AsyncOpenAI()
small_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
large_llm = ChatOpenAI(model="gpt-4o", temperature=0)  # 模擬 Gemini Pro

print("環境就緒")
print("  小模型（模擬 Gemma-7b）: gpt-4o-mini")
print("  大模型（模擬 Gemini Pro）: gpt-4o")

## Part 1：問題定義與成本計算

In [ ]:
# 量化路由的價值

DAILY_REQUESTS = 100_000
AVG_TOKENS = 2_000
SMALL_MODEL_COST = 0.05   # USD per 1M tokens（模擬 Gemma on GKE）
LARGE_MODEL_COST = 1.25   # USD per 1M tokens（Gemini Pro）
SIMPLE_RATIO = 0.70       # 70% 是簡單問題

def calculate_monthly_cost(simple_ratio: float) -> float:
    monthly_tokens = DAILY_REQUESTS * AVG_TOKENS * 30 / 1_000_000  # in millions
    small_cost = monthly_tokens * simple_ratio * SMALL_MODEL_COST
    large_cost = monthly_tokens * (1 - simple_ratio) * LARGE_MODEL_COST
    return small_cost + large_cost

no_routing_cost = calculate_monthly_cost(0)       # 全部用大模型
with_routing_cost = calculate_monthly_cost(SIMPLE_RATIO)
savings = (no_routing_cost - with_routing_cost) / no_routing_cost * 100

print("=" * 50)
print("路由系統的 ROI 計算")
print("=" * 50)
print(f"""
假設條件：
  每日請求: {DAILY_REQUESTS:,}
  平均 Token: {AVG_TOKENS:,}
  小模型成本: ${SMALL_MODEL_COST}/1M tokens
  大模型成本: ${LARGE_MODEL_COST}/1M tokens
  簡單問題比例: {SIMPLE_RATIO*100:.0f}%

月成本：
  無路由（全部 → 大模型）: ${no_routing_cost:,.0f}/月
  有路由（70% → 小模型）: ${with_routing_cost:,.0f}/月
  節省: {savings:.0f}%（約 ${no_routing_cost-with_routing_cost:,.0f}/月）

風險（路由器的代價）：
  如果路由誤判，把複雜問題送給小模型
  → 品質下滑 → 用戶不滿 → 業務影響 > 成本節省
""")

## Part 2：Layer 1 — Rule-based 快速路由（< 1ms）

In [ ]:
# Layer 1：明確規則的快速過濾
# 處理「一眼就看得出來」的情況，不需要 Embedding

import re

SIMPLE_PATTERNS = [
    r'^(你好|Hi|Hello|嗨|早安|午安|晚安)',
    r'^(謝謝|感謝|Thank)',
    r'^(什麼時間|現在幾點)',
]  # 明確的簡單問句 pattern

COMPLEX_SIGNALS = [
    "分析", "比較", "預測", "策略", "優化", "設計",
    "合約", "法律", "財務", "報告",
    "多步驟", "詳細說明", "深入",
    "analyze", "compare", "predict", "strategy"
]  # 複雜問題的信號詞

def layer1_router(query: str) -> Literal["small", "large", "uncertain"]:
    """
    Layer 1：Rule-based 路由（< 1ms）
    Returns: 'small'（送小模型）| 'large'（送大模型）| 'uncertain'（交 Layer 2）
    """
    query_lower = query.lower()
    
    # 規則 1：明確是複雜問題（高優先，送大模型）
    if len(query.split()) > 80:  # 超過 80 個詞，通常是複雜上下文
        return "large"
    
    for signal in COMPLEX_SIGNALS:
        if signal in query:
            return "large"
    
    # 規則 2：明確是簡單問題（送小模型）
    if len(query) < 30:  # 很短的問句
        for pattern in SIMPLE_PATTERNS:
            if re.match(pattern, query):
                return "small"
    
    # 無法確定，交給 Layer 2
    return "uncertain"


# 測試 Layer 1
test_queries_l1 = [
    ("你好！", "明確簡單"),
    ("分析這份合約的法律風險並給出建議", "明確複雜"),
    ("幫我把這段文字翻譯成英文", "不確定（短但不是問候）"),
    ("今天天氣如何", "不確定"),
]

print("Layer 1 路由測試：")
for query, expected in test_queries_l1:
    result = layer1_router(query)
    icon = {"small": "🔵", "large": "🔴", "uncertain": "⚪"}
    print(f"  {icon[result]} [{result:10s}] '{query}' （預期：{expected}）")

## Part 3：Layer 2 — Semantic Router（Embedding 相似度）

In [ ]:
# 黃金範例庫（路由器的核心資產）

SIMPLE_EXAMPLES = [
    "今天天氣怎樣？",
    "幫我把這段文字翻譯成英文",
    "這個 JSON 格式對嗎？",
    "請把這個清單整理一下",
    "幫我寫一個簡短的感謝訊息",
    "這個日期怎麼格式化？",
    "幫我把這段話改寫得更正式",
    "計算一下 15% 的折扣",
]

COMPLEX_EXAMPLES = [
    "分析這份合約的法律風險並給出修改建議",
    "根據這些銷售數據建立一個財務預測模型",
    "找出這段程式碼的 bug 並解釋根本原因",
    "比較這三個供應商的技術方案優劣",
    "設計一個 RAG 系統的架構，考慮到高可用性需求",
    "分析競爭對手的市場策略並提出應對方案",
    "請詳細說明 Transformer 的 Attention 機制",
    "評估這份投資計畫的 ROI 並列出風險",
]

print(f"黃金範例庫：")
print(f"  簡單範例: {len(SIMPLE_EXAMPLES)} 條")
print(f"  複雜範例: {len(COMPLEX_EXAMPLES)} 條")

In [ ]:
# ✅ Semantic Router 實作

class SemanticRouter:
    """基於 Embedding 相似度的語意路由器"""
    
    def __init__(self, threshold: float = 0.88):
        self.threshold = threshold
        self.simple_embeddings: list[list[float]] = []
        self.complex_embeddings: list[list[float]] = []
        self._initialized = False
    
    async def _get_embedding(self, text: str) -> list[float]:
        """呼叫 OpenAI Embedding API（生產中用 text-embedding-004）"""
        response = await client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    
    async def build_index(self, simple_examples: list[str], complex_examples: list[str]):
        """建立黃金範例的 Embedding 索引"""
        print("建立 Embedding 索引...")
        
        # 並行計算所有範例的 embedding
        all_texts = simple_examples + complex_examples
        response = await client.embeddings.create(
            model="text-embedding-3-small",
            input=all_texts
        )
        
        n_simple = len(simple_examples)
        all_embeddings = [e.embedding for e in response.data]
        self.simple_embeddings = all_embeddings[:n_simple]
        self.complex_embeddings = all_embeddings[n_simple:]
        self._initialized = True
        
        print(f"✅ 索引建立完成（{n_simple} 簡單 + {len(complex_examples)} 複雜）")
    
    def _cosine_similarity(self, a: list[float], b: list[float]) -> float:
        """計算餘弦相似度"""
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def _max_similarity(self, query_emb: list[float], examples: list[list[float]]) -> float:
        """查詢與範例庫的最高相似度"""
        return max(self._cosine_similarity(query_emb, ex) for ex in examples)
    
    async def route(self, query: str) -> dict:
        """路由決策，返回路由結果和相似度分數"""
        if not self._initialized:
            raise RuntimeError("Router 尚未初始化，請先呼叫 build_index()")
        
        # 計算 query 的 embedding
        query_emb = await self._get_embedding(query)
        
        # 計算與兩個類別的最高相似度
        simple_score = self._max_similarity(query_emb, self.simple_embeddings)
        complex_score = self._max_similarity(query_emb, self.complex_embeddings)
        
        # 路由決策
        if simple_score > self.threshold and simple_score > complex_score:
            decision = "small"
        elif complex_score > self.threshold and complex_score > simple_score:
            decision = "large"
        else:
            decision = "large"  # 保守策略：不確定時送大模型
        
        return {
            "decision": decision,
            "simple_score": round(simple_score, 4),
            "complex_score": round(complex_score, 4),
            "threshold": self.threshold
        }


# 初始化並建立索引
router = SemanticRouter(threshold=0.88)
await router.build_index(SIMPLE_EXAMPLES, COMPLEX_EXAMPLES)

In [ ]:
# ✅ 完整兩層路由器

async def hybrid_router(query: str) -> dict:
    """兩層混合路由器"""
    
    # Layer 1：Rule-based（< 1ms）
    l1_result = layer1_router(query)
    if l1_result != "uncertain":
        return {"decision": l1_result, "layer": 1, "simple_score": None, "complex_score": None}
    
    # Layer 2：Semantic Router（~50ms）
    l2_result = await router.route(query)
    l2_result["layer"] = 2
    return l2_result


# 測試查詢
test_queries = [
    "你好！今天天氣怎樣？",
    "幫我把這段話翻譯成英文：今天很開心",
    "分析這份合約的法律風險",
    "設計一個 RAG 系統應對高並發需求",
    "計算 1000 乘以 365",
    "評估這個投資策略的 ROI 並分析潛在風險",
    "幫我整理一份會議記錄摘要",  # 邊界情況
    "請詳細比較 LangGraph 和 CrewAI 的架構差異",
]

print("=" * 65)
print("兩層混合路由器測試")
print("=" * 65)
print(f"{'Query':<40} {'決策':<8} {'Layer':<7} {'Simple':>8} {'Complex':>8}")
print("-" * 65)

for q in test_queries:
    result = await hybrid_router(q)
    icon = "🔵" if result["decision"] == "small" else "🔴"
    q_short = q[:38] + ".." if len(q) > 40 else q
    simple_s = f"{result['simple_score']:.3f}" if result["simple_score"] else "N/A"
    complex_s = f"{result['complex_score']:.3f}" if result["complex_score"] else "N/A"
    print(f"  {q_short:<40} {icon}{result['decision']:<7} L{result['layer']:<6} {simple_s:>8} {complex_s:>8}")

## Part 4：Shadow Evaluation Pipeline（品質保障）

In [ ]:
# ✅ Shadow Evaluation：確保路由不犧牲品質
# 原理：把被路由到小模型的請求，同時也送給大模型，比較答案差距

async def shadow_evaluate(query: str, small_answer: str, large_answer: str) -> dict:
    """用 LLM-as-Judge 評估兩個答案的品質差距"""
    
    judge_prompt = f"""你是一個客觀的品質評審員。
評估以下兩個答案的品質差距。
答案 A 是小模型的回答，答案 B 是大模型（黃金標準）的回答。
以 JSON 輸出評估結果：
{{
  "quality_gap": "NONE|MINOR|MAJOR",
  "reason": "具體說明差異"
}}

問題：{query}

答案 A（小模型）：
{small_answer}

答案 B（大模型，黃金標準）：
{large_answer}
"""
    
    response = await small_llm.ainvoke([HumanMessage(content=judge_prompt)])
    
    try:
        raw = response.content
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"): raw = raw[4:]
        return json.loads(raw.strip())
    except:
        return {"quality_gap": "MINOR", "reason": response.content}


async def run_shadow_evaluation(queries: list[str], sample_rate: float = 1.0) -> dict:
    """
    對被路由到小模型的請求執行 Shadow Evaluation
    sample_rate: 抽樣比例（生產中通常 5~10%）
    """
    print(f"\nShadow Evaluation（抽樣率 {sample_rate*100:.0f}%）")
    
    results = []
    major_errors = 0
    
    for query in queries:
        # 路由決策
        route = await hybrid_router(query)
        if route["decision"] != "small":
            continue  # 只評估被路由到小模型的請求
        
        print(f"  評估：'{query[:40]}...'" if len(query) > 40 else f"  評估：'{query}'")
        
        # 並行取得兩個模型的答案
        small_task = small_llm.ainvoke([HumanMessage(content=query)])
        large_task = large_llm.ainvoke([HumanMessage(content=query)])
        small_resp, large_resp = await asyncio.gather(small_task, large_task)
        
        # 評估品質差距
        eval_result = await shadow_evaluate(query, small_resp.content, large_resp.content)
        gap = eval_result.get("quality_gap", "UNKNOWN")
        
        print(f"    品質差距: {gap} — {eval_result.get('reason', '')[:60]}")
        
        if gap == "MAJOR":
            major_errors += 1
        
        results.append({"query": query, "quality_gap": gap, "detail": eval_result})
    
    total_routed_to_small = len(results)
    error_rate = major_errors / total_routed_to_small if total_routed_to_small > 0 else 0
    
    summary = {
        "total_evaluated": total_routed_to_small,
        "major_errors": major_errors,
        "error_rate": round(error_rate, 3),
        "results": results
    }
    
    print(f"\n📊 Shadow Eval 摘要：")
    print(f"  路由到小模型的請求: {total_routed_to_small}")
    print(f"  品質差距 MAJOR: {major_errors}")
    print(f"  誤判率: {error_rate*100:.1f}%")
    
    if error_rate > 0.10:
        print(f"  ⚠️  誤判率 > 10%！建議調高路由閾值至 0.92")
    else:
        print(f"  ✅ 誤判率可接受")
    
    return summary


# 執行 Shadow Evaluation
print("=" * 50)
print("Shadow Evaluation Pipeline")
print("=" * 50)

# 使用較少的測試查詢（避免過多 API 呼叫）
shadow_test_queries = [
    "幫我把這段話翻譯成英文：今天很開心",
    "幫我整理一份會議記錄摘要",
    "今天天氣怎樣？",
]

eval_summary = await run_shadow_evaluation(shadow_test_queries)

## Part 5：動態閾值調整 + 面試白板語言

In [ ]:
# ✅ 動態閾值調整邏輯

def adjust_threshold(current_threshold: float, error_rate: float, 
                      query_category: str = "general") -> float:
    """
    根據 Shadow Eval 的誤判率動態調整路由閾值
    
    Logic:
    - 誤判率 > 10%：調高閾值（更保守，送更多到大模型）
    - 誤判率 < 2%：可以小幅降低閾值（省更多成本）
    """
    new_threshold = current_threshold
    
    if error_rate > 0.10:  # 誤判率 > 10%
        adjustment = 0.04  # 閾值提高 0.04
        new_threshold = min(0.99, current_threshold + adjustment)
        action = f"↑ 調高（誤判率 {error_rate*100:.0f}% > 10%）"
    elif error_rate < 0.02:  # 誤判率 < 2%
        adjustment = 0.01  # 閾值小幅降低 0.01
        new_threshold = max(0.70, current_threshold - adjustment)
        action = f"↓ 小幅降低（誤判率 {error_rate*100:.0f}% < 2%）"
    else:
        action = "維持不變（誤判率在可接受範圍）"
    
    return new_threshold, action


# 模擬不同場景的閾值調整
scenarios = [
    (0.88, 0.15, "合約相關問題"),   # 誤判率過高
    (0.88, 0.05, "一般問答"),       # 可接受範圍
    (0.88, 0.01, "簡單格式轉換"),   # 可以放寬
]

print("動態閾值調整：")
print("-" * 60)
for threshold, error_rate, category in scenarios:
    new_threshold, action = adjust_threshold(threshold, error_rate, category)
    print(f"  [{category}]")
    print(f"    誤判率: {error_rate*100:.0f}% | 閾值: {threshold} → {new_threshold} | {action}")

In [ ]:
print("""
面試答題框架：
─────────────────────────────────────────────────────
設計混合路由系統要解決兩個問題：準確路由 + 品質保障。

路由架構用兩層：
  Layer 1: Rule-based 快速過濾（< 1ms）
    - 請求長度、信號詞、Pattern Match 處理明顯的情況
    - 不確定的進入 Layer 2
  
  Layer 2: Semantic Router（~50ms）
    - 輕量 Embedding 模型計算當前請求和黃金範例的相似度
    - 超過 0.88 閾值的送小模型
    - 不確定的保守策略送大模型

保守偏誤原則：
  誤判方向不對稱：
  把複雜問題送給小模型（假負例）→ 品質差，用戶不滿 ← 嚴重
  把簡單問題送給大模型（假正例）→ 浪費成本，用戶滿意 ← 輕微
  所以：不確定時，送大模型

品質保障：Shadow Evaluation
  每日從生產日誌抽 5%，做 Shadow Run
  同一請求同時跑小模型和大模型，比較答案差距
  計算誤判率；如果某類 Query 誤判率 > 10%，
  自動調高路由閾值，確保那類問題更容易被判為複雜

量化成本節省（面試要說得出數字）：
  10 萬請求/天，從 $7,500/月降到 $3,790/月，節省 49%
─────────────────────────────────────────────────────
""")